In [1]:
import numpy as np
import scipy


from metastable.state import FixedPointMap
from metastable.eom import EOM, Params
from metastable.manifold_inverses import calculate_manifold_inverses
from metastable.incoming_quantum_vector import extend_to_keldysh_state


fixed_point_map = FixedPointMap.load("map.npz")
epsilon_idx = -1
kappa_idx = 20
params = Params(
    epsilon=fixed_point_map.epsilon_linspace[epsilon_idx],
    kappa=fixed_point_map.kappa_linspace[kappa_idx],
    delta=fixed_point_map.delta,
    chi=fixed_point_map.chi,
)
eom = EOM(params=params)
classical_saddle_point = fixed_point_map.fixed_points[epsilon_idx, kappa_idx, 0]
classical_focus_point = fixed_point_map.fixed_points[epsilon_idx, kappa_idx, 1]
print(params)
print(classical_saddle_point, classical_focus_point)


keldysh_saddle_point = extend_to_keldysh_state(classical_saddle_point)
keldysh_focus_point = extend_to_keldysh_state(classical_focus_point)
_, saddle_point_unstable_manifold_inverse = calculate_manifold_inverses(
    keldysh_saddle_point, params
)
focus_point_stable_manifold_inverse, _ = calculate_manifold_inverses(
    keldysh_focus_point, params
)

old_vectors = np.load("bright-to-unstable.npz")

In [2]:
keldysh_fixed_point = keldysh_focus_point
eom = EOM(params=params)
jacobian = eom.jacobian_func(keldysh_fixed_point)
eigenvalues, eigenvectors = np.linalg.eig(jacobian)
stable_manifold_mask = eigenvalues <= 0
unstable_manifold_mask = eigenvalues > 0
stable_manifold = eigenvectors[:, stable_manifold_mask]
unstable_manifold = eigenvectors[:, unstable_manifold_mask]
inverse_eigenvectors = np.linalg.inv(eigenvectors)
inverse_unstable_manifold = inverse_eigenvectors[unstable_manifold_mask, :]
inverse_stable_manifold = inverse_eigenvectors[stable_manifold_mask, :]

In [3]:
eigenvalues

In [4]:
eigenvectors[:,0]

In [5]:
eigenvectors[:,1]

In [6]:
np.exp(-1j*np.angle(eigenvectors[1][0]))*eigenvectors[1]

In [7]:
focus_point_alt_vectors = old_vectors["vectors_orthogonal_to_stable_point_outgoing_vectors"].reshape([2, 4])

In [8]:
focus_point_alt_vectors

In [9]:
focus_point_stable_manifold_inverse

In [10]:
np.swapaxes(old_vectors["vectors_orthogonal_to_unstable_point_incoming_vectors"], 0, 1)

In [11]:
old_vectors["vectors_orthogonal_to_unstable_point_incoming_vectors"].reshape([2, 4])

In [12]:
saddle_point_unstable_manifold_inverse

In [13]:
eom = EOM(params=params)
jacobian = eom.jacobian_func(keldysh_focus_point)
eigenvalues, eigenvectors = np.linalg.eig(jacobian)

In [14]:
eigenvalues

In [15]:
np.dot(old_vectors["vectors_orthogonal_to_unstable_point_incoming_vectors"].reshape([2, 4]), eigenvectors[:,0])

In [16]:
print(np.dot(eigenvectors[:,1], old_vectors["vectors_orthogonal_to_unstable_point_incoming_vectors"]))
print(np.dot(eigenvectors[:,2], old_vectors["vectors_orthogonal_to_unstable_point_incoming_vectors"]))

In [17]:
print(np.dot(eigenvectors[:,1], old_vectors["vectors_orthogonal_to_unstable_point_incoming_vectors"]))
print(np.dot(eigenvectors[:,2], old_vectors["vectors_orthogonal_to_unstable_point_incoming_vectors"]))

In [18]:
eigenvalues

In [19]:
np.dot(jacobian, eigenvectors[:,1]) / eigenvalues[1]

In [20]:
eigenvectors[:,1]

In [21]:
np.swapaxes(old_vectors["vectors_orthogonal_to_stable_point_outgoing_vectors"], 0, 1)

In [22]:
focus_point_stable_manifold_inverse

In [23]:
test = np.vstack([focus_point_stable_manifold_inverse[0] + focus_point_stable_manifold_inverse[1], focus_point_stable_manifold_inverse[0] - focus_point_stable_manifold_inverse[1]])

In [27]:
def bc(ya, yb):
    return np.hstack(
        [
            np.abs(np.dot(focus_point_stable_manifold_inverse, ya - keldysh_focus_point)),
            np.abs(np.dot(saddle_point_unstable_manifold_inverse, yb - keldysh_saddle_point)),
        ]
    )


print(keldysh_saddle_point, keldysh_focus_point)

t_guess = np.linspace(0.0, 8.0, 10001)
y_guess = (
    keldysh_focus_point[:, np.newaxis]
    + t_guess[np.newaxis, :]
    * (keldysh_saddle_point - keldysh_focus_point)[:, np.newaxis]
    / t_guess[-1]
)
wrapper = lambda x, y: eom.y_dot_func(y)
res = scipy.integrate.solve_bvp(
    wrapper, bc, t_guess, y_guess, tol=3e-14, max_nodes=200000
)


import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,1,figsize=(5, 5))
t_plot = np.linspace(0, t_guess[-1], 1001)
y0_plot = res.sol(t_plot)[0]
y1_plot = res.sol(t_plot)[1]
axes.plot(y0_plot,y1_plot)
plt.show()


In [28]:
def bc(ya, yb):
    return np.hstack(
        [
            np.abs(np.dot(test, ya - keldysh_focus_point)),
            np.abs(np.dot(saddle_point_unstable_manifold_inverse, yb - keldysh_saddle_point)),
        ]
    )


print(keldysh_saddle_point, keldysh_focus_point)

t_guess = np.linspace(0.0, 8.0, 10001)
y_guess = (
    keldysh_focus_point[:, np.newaxis]
    + t_guess[np.newaxis, :]
    * (keldysh_saddle_point - keldysh_focus_point)[:, np.newaxis]
    / t_guess[-1]
)
wrapper = lambda x, y: eom.y_dot_func(y)
res = scipy.integrate.solve_bvp(
    wrapper, bc, t_guess, y_guess, tol=3e-14, max_nodes=200000
)


import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,1,figsize=(5, 5))
t_plot = np.linspace(0, t_guess[-1], 1001)
y0_plot = res.sol(t_plot)[0]
y1_plot = res.sol(t_plot)[1]
axes.plot(y0_plot,y1_plot)
plt.show()


In [29]:
def bc(ya, yb):
    return np.hstack(
        [
            np.abs(np.dot(np.swapaxes(old_vectors["vectors_orthogonal_to_stable_point_outgoing_vectors"], 0, 1), ya - keldysh_focus_point)),
            np.abs(np.dot(saddle_point_unstable_manifold_inverse, yb - keldysh_saddle_point)),
        ]
    )


print(keldysh_saddle_point, keldysh_focus_point)

t_guess = np.linspace(0.0, 8.0, 10001)
y_guess = (
    keldysh_focus_point[:, np.newaxis]
    + t_guess[np.newaxis, :]
    * (keldysh_saddle_point - keldysh_focus_point)[:, np.newaxis]
    / t_guess[-1]
)
wrapper = lambda x, y: eom.y_dot_func(y)
res = scipy.integrate.solve_bvp(
    wrapper, bc, t_guess, y_guess, tol=3e-14, max_nodes=200000
)


import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,1,figsize=(5, 5))
t_plot = np.linspace(0, t_guess[-1], 1001)
y0_plot = res.sol(t_plot)[0]
y1_plot = res.sol(t_plot)[1]
axes.plot(y0_plot,y1_plot)
plt.show()


In [30]:
def bc(ya, yb):
    return np.hstack(
        [
            np.abs(np.dot(np.swapaxes(old_vectors["vectors_orthogonal_to_stable_point_outgoing_vectors"], 0, 1), ya - keldysh_focus_point)),
            np.abs(np.dot(np.swapaxes(old_vectors["vectors_orthogonal_to_unstable_point_incoming_vectors"], 0, 1), yb - keldysh_saddle_point)),
        ]
    )


print(keldysh_saddle_point, keldysh_focus_point)

t_guess = np.linspace(0.0, 8.0, 10001)
y_guess = (
    keldysh_focus_point[:, np.newaxis]
    + t_guess[np.newaxis, :]
    * (keldysh_saddle_point - keldysh_focus_point)[:, np.newaxis]
    / t_guess[-1]
)
wrapper = lambda x, y: eom.y_dot_func(y)
res = scipy.integrate.solve_bvp(
    wrapper, bc, t_guess, y_guess, tol=3e-14, max_nodes=200000
)


import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,1,figsize=(5, 5))
t_plot = np.linspace(0, t_guess[-1], 1001)
y0_plot = res.sol(t_plot)[0]
y1_plot = res.sol(t_plot)[1]
axes.plot(y0_plot,y1_plot)
plt.show()


In [31]:
# def bc(ya, yb):
#     return np.hstack(
#         [
#             np.abs(np.dot(old_vectors["vectors_orthogonal_to_stable_point_outgoing_vectors"].reshape([2, 4]), ya - keldysh_focus_point)),
#             np.abs(np.dot(saddle_point_unstable_manifold_inverse, yb - keldysh_saddle_point)),
#         ]
#     )

def bc(ya, yb):
    return np.hstack(
        [
            np.abs(np.dot(np.swapaxes(old_vectors["vectors_orthogonal_to_stable_point_outgoing_vectors"], 0, 1), ya - keldysh_focus_point)),
            np.abs(np.dot(old_vectors["vectors_orthogonal_to_unstable_point_incoming_vectors"].reshape([2, 4]), yb - keldysh_saddle_point)),
        ]
    )

# def bc(ya, yb):
#     return np.hstack(
#         [
#             np.dot(focus_point_stable_manifold_inverse, ya - keldysh_focus_point),
#             np.dot(saddle_point_unstable_manifold_inverse, yb - keldysh_saddle_point),
#         ]
#     )


print(keldysh_saddle_point, keldysh_focus_point)

t_guess = np.linspace(0.0, 8.0, 10001)
y_guess = (
    keldysh_focus_point[:, np.newaxis]
    + t_guess[np.newaxis, :]
    * (keldysh_saddle_point - keldysh_focus_point)[:, np.newaxis]
    / t_guess[-1]
)
wrapper = lambda x, y: eom.y_dot_func(y)
res = scipy.integrate.solve_bvp(
    wrapper, bc, t_guess, y_guess, tol=3e-14, max_nodes=200000
)


import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,1,figsize=(5, 5))
t_plot = np.linspace(0, t_guess[-1], 1001)
y0_plot = res.sol(t_plot)[0]
y1_plot = res.sol(t_plot)[1]
axes.plot(y0_plot,y1_plot)
plt.show()


In [40]:
# def bc(ya, yb):
#     return np.hstack(
#         [
#             np.abs(np.dot(old_vectors["vectors_orthogonal_to_stable_point_outgoing_vectors"].reshape([2, 4]), ya - keldysh_focus_point)),
#             np.abs(np.dot(saddle_point_unstable_manifold_inverse, yb - keldysh_saddle_point)),
#         ]
#     )

def bc(ya, yb):
    return np.hstack(
        [
            np.abs(np.dot(old_vectors["vectors_orthogonal_to_stable_point_outgoing_vectors"].reshape([2, 4]), ya - keldysh_focus_point)),
            np.abs(np.dot(old_vectors["vectors_orthogonal_to_unstable_point_incoming_vectors"].reshape([2, 4]), yb - keldysh_saddle_point)),
        ]
    )

# def bc(ya, yb):
#     return np.hstack(
#         [
#             np.dot(focus_point_stable_manifold_inverse, ya - keldysh_focus_point),
#             np.dot(saddle_point_unstable_manifold_inverse, yb - keldysh_saddle_point),
#         ]
#     )


print(keldysh_saddle_point, keldysh_focus_point)

t_guess = np.linspace(0.0, 8.0, 10001)
y_guess = (
    keldysh_focus_point[:, np.newaxis]
    + t_guess[np.newaxis, :]
    * (keldysh_saddle_point - keldysh_focus_point)[:, np.newaxis]
    / t_guess[-1]
)
wrapper = lambda x, y: eom.y_dot_func(y)
res = scipy.integrate.solve_bvp(
    wrapper, bc, t_guess, y_guess, tol=3e-14, max_nodes=200000
)


import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,1,figsize=(5, 5))
t_plot = np.linspace(0, t_guess[-1], 1001)
y0_plot = res.sol(t_plot)[0]
y1_plot = res.sol(t_plot)[1]
axes.plot(y0_plot,y1_plot)
plt.show()
